# Tutorial: Interconnection Financing Triage

## What is this notebook?

This notebook is a step-by-step tutorial for analyzing EU cross-border electricity interconnection projects and classifying them into **financing tracks**. The goal is to estimate how much of the ~170 bn EUR TYNDP investment pipeline can be market-financed vs. how much needs targeted EU support.

### The three financing tracks

| Track | Description | Financing mix |
|---|---|---|
| **Track 1** | Commercially viable — congestion rents cover costs | Private capital (merchant/hybrid) |
| **Track 2** | Not commercially viable, but TSOs are creditworthy | Regulated tariffs + CBCA |
| **Track 3** | Not commercially viable AND TSO is credit-constrained | CEF grants + EIB loans |

### How to use this notebook

Run each cell in order. Each section explains **what** is being computed and **why**, so you can follow the analytical logic even if you're not familiar with the codebase. All business logic lives in the `grid_financing` Python package — this notebook only orchestrates and displays results.

## Step 0: Setup

We import the `grid_financing` package and configure display settings. The package handles all data loading, calculations, and classification.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

from grid_financing import (
    build_aggregate_summary,
    build_project_master_table,
    build_sensitivity_cases,
    calculate_project_metrics,
    classify_projects,
    export_outputs,
    manual_source_status,
)
from grid_financing.exports import build_triage_scatter_dataset, build_financing_stack_dataset
from grid_financing.classification import TRACK_1, TRACK_2, TRACK_3

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Use notebook_connected renderer (loads plotly.js from CDN, works in VS Code and Jupyter)
pio.renderers.default = "notebook_connected"

## Step 1: Check data sources

Before loading anything, let's verify which input files are available. The pipeline requires:

- **TYNDP 2024 workbook** — the main project list, investment-level CAPEX, and CBA welfare scenarios
- **TYNDP 2022 workbook** — older CBA scenarios used as fallback
- **Local hourly prices** — country-level day-ahead prices stored in `data/european_wholesale_electricity_price_data_hourly/all_countries.csv`
- **Manual CSV tables** — TSO credit ratings, project participant CAPEX splits, border-zone mappings

This notebook uses the local hourly CSV already in the repository. It does **not** call an external API in the current workflow. Sources marked `exists=False` will not crash the pipeline — affected projects are flagged with data-quality issues instead.


In [2]:
source_status = pd.DataFrame(manual_source_status())
source_status

,source_id,kind,exists,variant,path,message
0,tyndp2022_workbook,local,True,legacy,/Users/lucas/Workspace/PolicyBrief/GridFinanci...,
1,tyndp2024_workbook,local,True,legacy,/Users/lucas/Workspace/PolicyBrief/GridFinanci...,
2,local_hourly_prices,local,True,canonical,/Users/lucas/Workspace/PolicyBrief/GridFinanci...,
3,tso_credit_reference,manual-download-required,True,canonical,/Users/lucas/Workspace/PolicyBrief/GridFinanci...,
4,project_participants,manual-download-required,True,canonical,/Users/lucas/Workspace/PolicyBrief/GridFinanci...,
5,project_overrides,manual-download-required,False,missing,,Missing `data/manual/project_overrides.csv`. A...
6,border_zone_map,manual-download-required,True,canonical,/Users/lucas/Workspace/PolicyBrief/GridFinanci...,
7,dayahead_price_inputs,manual-download-required,False,missing,,Missing `data/manual/dayahead_price_inputs.csv...


## Step 2: Build the project master table

This is the core data assembly step. It merges multiple sources into one row per cross-border project in the base case:

1. **Load projects** from the TYNDP 2024 `Trans.Projects` sheet (filters to cross-border only)
2. **Aggregate CAPEX** from `Trans.Investments` (sums investment-level costs per project)
3. **Merge CBA welfare values** from 2024 and 2022 scenario tabs
4. **Apply manual overrides** for missing transfer capacities and country mappings
5. **Attach credit data** — TSO ratings, RAB sizes, sovereign fiscal indicators, cohesion status
6. **Preprocess hourly prices** — in development mode, build border spreads from overlapping hourly observations in the local CSV and keep the latest full year for the base case
7. **Flag data-quality issues** — missing CAPEX, missing prices, unresolved multi-country projects, etc.

The year-by-year hourly preprocessing can also expand the table into multiple full-year price scenarios for sensitivity work, for example `proxy_2020` through `proxy_2025`.


In [3]:
master = build_project_master_table(development_mode=True)

print(f"Cross-border projects loaded: {master['project_id'].nunique()}")
print(f"Project rows in base case: {len(master)}")
print(f"Projects with data-quality flags: {master['has_data_quality_issue'].sum()}")
print(f"Price input mode: {master['price_input_mode'].iloc[0]}")
print()

preview = master[[
    "project_id", "project_name", "countries", "status",
    "capex_meur", "capacity_mw",
    "price_scenario", "data_year", "hourly_observation_count",
    "avg_price_diff_eur_per_mwh", "data_quality_flags",
]].head(10)

data_issue_summary = (
    master.loc[master["has_data_quality_issue"], "data_quality_flags"]
    .dropna()
    .str.split(";")
    .explode()
    .value_counts()
    .rename_axis("issue")
    .reset_index(name="project_count")
)

print("Base-case preview:")
display(preview)
print("Data issue summary:")
display(data_issue_summary)


Cross-border projects loaded: 100
Project rows in base case: 100
Projects with data-quality flags: 47
Price input mode: development-local-proxy

Base-case preview:


,project_id,project_name,countries,status,capex_meur,capacity_mw,price_scenario,data_year,hourly_observation_count,avg_price_diff_eur_per_mwh,data_quality_flags
0,4.0,Interconnection Portugal-Spain,ES ; PT,Construction,111.18,1500.0,proxy_2025,2025.0,8760.0,1.347471,<NA>
1,16.0,Biscay Gulf,ES ; FR,Construction,3100.00,2200.0,proxy_2025,2025.0,8760.0,20.325945,<NA>
2,28.0,Italy-Montenegro,IT ; ME,Construction,424.00,600.0,proxy_2024,2024.0,8784.0,19.686683,missing_credit_reference
3,29.0,Italy-Tunisia,IT ; TN,Construction,850.00,600.0,historical_proxy,NaN,NaN,NaN,missing_credit_reference;missing_price_inputs
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,BE ; DE ; LU,Under Consideration,210.00,500.0,proxy_2025,2025.0,8760.0,12.475568,missing_credit_reference
5,47.0,Westtirol (AT) - Vöhringen (DE),AT ; DE,Planning,10.30,600.0,proxy_2025,2025.0,8760.0,13.041811,<NA>
6,81.0,North South Interconnector,GB ; IE,Permitting,363.00,950.0,proxy_2025,2025.0,8760.0,31.960582,missing_credit_reference
7,94.0,GerPol Improvements,DE ; PL,Construction,110.80,1500.0,proxy_2025,2025.0,8760.0,19.908811,<NA>
8,107.0,Celtic Interconnector,FR ; IE,Construction,1623.00,700.0,proxy_2025,2025.0,8760.0,58.082207,<NA>
9,111.0,Aurora line (3rd AC Finland-Sweden north),FI ; SE,Construction,270.00,900.0,proxy_2025,2025.0,8760.0,21.458082,<NA>


Data issue summary:


,issue,project_count
0,missing_credit_reference,45
1,missing_price_inputs,20
2,missing_transfer_capacity,6


### Step 2.1: Explore the pipeline — CAPEX by project status

Before running the triage, let's understand what's in the TYNDP pipeline. The 100 cross-border projects span four development stages:

- **Under Consideration** — earliest stage, no permitting yet
- **Planning** — in planning but not yet in permitting
- **Permitting** — actively going through permitting
- **Construction** — already under construction, likely already financed

**Projects "Under Construction" (~14 bn EUR) are likely already financed**, so the remaining ~156 bn EUR is the relevant financing gap — consistent with the ~150 bn figure cited in the policy brief.

In [4]:
# CAPEX breakdown by project development status
status_summary = (
    master.groupby("status")
    .agg(
        project_count=("project_id", "nunique"),
        total_capex_meur=("capex_meur", "sum"),
    )
    .sort_values("total_capex_meur", ascending=False)
)
status_summary["total_capex_beur"] = status_summary["total_capex_meur"] / 1000
status_summary["share_pct"] = (
    status_summary["total_capex_meur"] / status_summary["total_capex_meur"].sum() * 100
).round(1)

print(f"Total pipeline: {status_summary['total_capex_meur'].sum()/1000:.1f} bn EUR "
      f"across {status_summary['project_count'].sum()} projects")
print(f"Excluding Construction: "
      f"{(status_summary['total_capex_meur'].sum() - status_summary.loc['Construction', 'total_capex_meur'])/1000:.1f} bn EUR")
print()
status_summary[["project_count", "total_capex_beur", "share_pct"]]

Total pipeline: 169.9 bn EUR across 100 projects
Excluding Construction: 156.1 bn EUR



,project_count,total_capex_beur,share_pct
status,,,
Under Consideration,40,123.214084,72.5
Planning,29,23.561613,13.9
Construction,12,13.790713,8.1
Permitting,19,9.330541,5.5


In [5]:
# Visualize CAPEX by status
fig_status = px.bar(
    status_summary.reset_index(),
    x="status",
    y="total_capex_beur",
    text="project_count",
    title="CAPEX Pipeline by Project Status (bn EUR)",
    labels={"total_capex_beur": "Total CAPEX (bn EUR)", "status": "Development status"},
    color="status",
    category_orders={"status": ["Under Consideration", "Planning", "Permitting", "Construction"]},
)
fig_status.update_traces(texttemplate="%{text} projects", textposition="outside")
fig_status.update_layout(showlegend=False, height=450)
fig_status.show()

### Step 2.2: Top projects by CAPEX

The pipeline is heavily concentrated — a few very large projects dominate the total. Let's see which projects carry the most investment weight.

In [6]:
# Top 15 projects by CAPEX
top_projects = master.nlargest(15, "capex_meur")[[
    "project_id", "project_name", "countries", "status",
    "capex_meur", "capacity_mw",
]].copy()
top_projects["capex_beur"] = top_projects["capex_meur"] / 1000

print(f"Top 15 projects account for {top_projects['capex_meur'].sum()/1000:.1f} bn EUR "
      f"({top_projects['capex_meur'].sum() / master['capex_meur'].sum() * 100:.0f}% of total)")
print()
top_projects

Top 15 projects account for 110.1 bn EUR (65% of total)



,project_id,project_name,countries,status,capex_meur,capacity_mw,capex_beur
46,335.0,Project 335 - North Sea Wind Power Hub,DE ; DK ; NL,Under Consideration,19571.0,None,19.5710
91,1215.0,NaN,DE ; MA,Under Consideration,14480.0,3600.0,14.4800
66,1092.0,Offshore Hybrid HVDC Interconnector BEDK,BE ; DK,Planning,10092.0,2000.0,10.0920
96,1231.0,NaN,DE ; GR,Under Consideration,8100.0,3000.0,8.1000
85,1208.0,NaN,DZ ; IT ; TN,Under Consideration,7757.0,2000.0,7.7570
92,1216.0,NaN,HU ; RO,Under Consideration,7284.0,2500.0,7.2840
81,1200.0,NaN,BE ; DE ; DK ; NO,Under Consideration,6920.0,1400.0,6.9200
88,1211.0,NaN,DE ; EE ; LV,Under Consideration,6600.0,2000.0,6.6000
90,1214.0,NaN,DE ; DK,Under Consideration,6000.0,None,6.0000
78,1192.0,NaN,DE ; GB,Under Consideration,5263.0,None,5.2630


## Step 3: Calculate financing metrics

Now we compute the ratios that drive the triage classification. Here's what each metric means:

| Metric | Formula | Interpretation |
|---|---|---|
| **Capital Recovery Factor** | `r / (1 - (1+r)^(-n))` | Annual repayment as fraction of initial cost (r=5%, n=40yr -> CRF≈0.058) |
| **Annualized CAPEX** | `CAPEX × CRF` | Yearly cost the project must cover to break even |
| **Congestion rent** | `Σ_t |price_a,t - price_b,t| × capacity × utilization / 1M` | Revenue estimated directly from overlapping hourly spreads in the selected full year |
| **Commercial ratio** | `congestion_rent / annualized_CAPEX` | >0.7 means the project can largely self-finance |
| **Social BCR** | `ΔSEW / annualized_CAPEX` | Welfare benefit per euro of cost (from TYNDP CBA) |
| **Credit constraint score** | `project_CAPEX_share / TSO_RAB` | How large the project is relative to the sponsoring TSO's balance sheet |

The key change is that congestion rent is now built from hourly observations first; the notebook no longer relies on a standalone `× 8760` assumption when hourly data are available.


In [7]:
metrics = calculate_project_metrics(master)

print(f"CRF at 5% / 40yr: {metrics['capital_recovery_factor'].iloc[0]:.4f}")
print(f"Projects with computable commercial ratio: {metrics['commercial_ratio'].notna().sum()} / {len(metrics)}")
print()

metrics[[
    "project_id", "project_name", "price_scenario", "data_year",
    "hourly_observation_count", "congestion_rent_basis",
    "annualized_capex_meur_per_year",
    "estimated_congestion_rent_meur_per_year",
    "commercial_ratio", "social_bcr",
    "credit_constraint_score", "credit_constrained",
]].head(15)


CRF at 5% / 40yr: 0.0583
Projects with computable commercial ratio: 76 / 100



,project_id,project_name,price_scenario,data_year,hourly_observation_count,congestion_rent_basis,annualized_capex_meur_per_year,estimated_congestion_rent_meur_per_year,commercial_ratio,social_bcr,credit_constraint_score,credit_constrained
0,4.0,Interconnection Portugal-Spain,proxy_2025,2025.0,8760.0,hourly_price_sum,6.479366,10.623465,1.639584,4.012738,0.015883,False
1,16.0,Biscay Gulf,proxy_2025,2025.0,8760.0,hourly_price_sum,180.662300,235.03297,1.300952,2.313709,0.166667,True
2,28.0,Italy-Montenegro,proxy_2024,2024.0,8784.0,hourly_price_sum,24.709940,62.254015,2.519392,0.242817,0.009422,False
3,29.0,Italy-Tunisia,historical_proxy,NaN,NaN,hourly_price_sum,49.536437,NaN,NaN,1.009358,0.018889,False
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,proxy_2025,2025.0,8760.0,hourly_price_sum,12.238414,32.785794,2.678925,1.797619,NaN,False
5,47.0,Westtirol (AT) - Vöhringen (DE),proxy_2025,2025.0,8760.0,hourly_price_sum,0.600265,41.128654,68.517487,58.307575,0.001717,True
6,81.0,North South Interconnector,proxy_2025,2025.0,8760.0,hourly_price_sum,21.154973,159.585579,7.543644,0.189081,0.051857,True
7,94.0,GerPol Improvements,proxy_2025,2025.0,8760.0,hourly_price_sum,6.457220,156.961062,24.307838,21.681156,0.011080,True
8,107.0,Celtic Interconnector,proxy_2025,2025.0,8760.0,hourly_price_sum,94.585456,213.696055,2.259291,2.569105,0.231857,True
9,111.0,Aurora line (3rd AC Finland-Sweden north),proxy_2025,2025.0,8760.0,hourly_price_sum,15.735104,101.505312,6.450883,2.923400,0.038571,True


## Step 4: Classify projects into financing tracks

The classification logic is straightforward:

1. **Commercial ratio > 0.7?** → **Track 1** (market-financed). Congestion rents are high enough for merchant or hybrid financing.
2. **Not commercially viable, but TSO is creditworthy?** → **Track 2** (regulated + CBCA). The TSO can finance through regulated tariffs with a Cross-Border Cost Allocation agreement.
3. **Not commercially viable AND credit-constrained?** → **Track 3** (needs EU support). The TSO's balance sheet is too small relative to project cost, or the sovereign is fiscally stressed. These need CEF grants + EIB loans.
4. **Missing data?** → **Unclassified**. Projects without price data or transfer capacity can't be evaluated.

A project is "credit-constrained" if **any** of these hold for **either side**:
- Credit constraint score > 0.15 (project is >15% of TSO RAB)
- TSO has sub-investment-grade rating
- Sovereign has debt >100% GDP AND deficit >3% GDP

After classification, the financing stack is estimated per project. Track 3 CEF grant rates default to 85% for cohesion-country sides and 50% otherwise.

In [8]:
classified = classify_projects(metrics)

print("Track distribution:")
print(classified["financing_track"].value_counts().to_string())
print()

classified[[
    "project_id", "project_name", "countries",
    "capex_meur", "commercial_ratio", "social_bcr",
    "credit_constraint_score", "financing_track",
    "estimated_cef_grant_meur", "data_quality_flags",
]].head(15)

Track distribution:
financing_track
Track 1: Market-financed (merchant/hybrid)           67
Unclassified: insufficient data                      24
Track 3: Credit-constrained - targeted EU support     8
Track 2: Regulated + CBCA                             1



,project_id,project_name,countries,capex_meur,commercial_ratio,social_bcr,credit_constraint_score,financing_track,estimated_cef_grant_meur,data_quality_flags
0,4.0,Interconnection Portugal-Spain,ES ; PT,111.18,1.639584,4.012738,0.015883,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
1,16.0,Biscay Gulf,ES ; FR,3100.00,1.300952,2.313709,0.166667,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
2,28.0,Italy-Montenegro,IT ; ME,424.00,2.519392,0.242817,0.009422,Track 1: Market-financed (merchant/hybrid),0.0,missing_credit_reference
3,29.0,Italy-Tunisia,IT ; TN,850.00,NaN,1.009358,0.018889,Unclassified: insufficient data,0.0,missing_credit_reference;missing_price_inputs
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,BE ; DE ; LU,210.00,2.678925,1.797619,NaN,Track 1: Market-financed (merchant/hybrid),0.0,missing_credit_reference
5,47.0,Westtirol (AT) - Vöhringen (DE),AT ; DE,10.30,68.517487,58.307575,0.001717,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
6,81.0,North South Interconnector,GB ; IE,363.00,7.543644,0.189081,0.051857,Track 1: Market-financed (merchant/hybrid),0.0,missing_credit_reference
7,94.0,GerPol Improvements,DE ; PL,110.80,24.307838,21.681156,0.011080,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
8,107.0,Celtic Interconnector,FR ; IE,1623.00,2.259291,2.569105,0.231857,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
9,111.0,Aurora line (3rd AC Finland-Sweden north),FI ; SE,270.00,6.450883,2.923400,0.038571,Track 1: Market-financed (merchant/hybrid),0.0,<NA>


## Step 5: Visualize the triage results

### 5.1 Scatter plot — the core policy chart

This is the centrepiece visualization for the policy brief. Each dot is a project:

- **X-axis:** commercial viability ratio. Right of the dashed line at 0.7 = can self-finance.
- **Y-axis:** credit constraint score. Above the dashed line at 0.15 = TSO balance sheet is strained.
- **Dot size:** total CAPEX.
- **Color:** financing track.

The quadrants map directly to tracks:
- **Right half** → Track 1 (commercially viable, any credit profile)
- **Bottom-left** → Track 2 (not viable, but creditworthy TSO)
- **Top-left** → Track 3 (not viable AND credit-constrained)

In [9]:
scatter_df = build_triage_scatter_dataset(classified)

fig_scatter = px.scatter(
    scatter_df,
    x="commercial_ratio",
    y="credit_constraint_score",
    size="capex_meur",
    color="financing_track",
    hover_name="project_name",
    title="Financing Triage: Commercial Viability vs. Credit Constraint",
    labels={
        "commercial_ratio": "Commercial viability ratio",
        "credit_constraint_score": "Credit constraint score (CAPEX share / TSO RAB)",
    },
)
fig_scatter.add_vline(x=0.7, line_dash="dash", line_color="gray",
                       annotation_text="Merchant threshold (0.7)")
fig_scatter.add_hline(y=0.15, line_dash="dash", line_color="gray",
                       annotation_text="Credit constraint threshold (0.15)")
fig_scatter.update_layout(height=600)
fig_scatter.show()

### 5.2 Aggregate financing needs

How does the ~170 bn EUR pipeline break down by financing track and source?

The key policy insight: **if merchant financing works for Track 1 projects, CEF resources can be concentrated on the credit-constrained projects (Track 3) that need them most.**

In [10]:
summary = build_aggregate_summary(classified)
summary[[
    "financing_track", "project_count",
    "total_capex_beur", "total_cef_beur", "total_eib_beur",
    "total_private_beur", "total_tso_beur",
]]

,financing_track,project_count,total_capex_beur,total_cef_beur,total_eib_beur,total_private_beur,total_tso_beur
0,Track 1: Market-financed (merchant/hybrid),67,72.927619,0.000000,0.000000,72.927619,0.000000
1,Track 2: Regulated + CBCA,1,0.284000,0.085200,0.071000,0.000000,0.127800
2,Track 3: Credit-constrained - targeted EU support,8,22.949190,14.589735,4.589838,0.000000,4.214637
3,Unclassified: insufficient data,24,73.736142,0.000000,0.000000,0.000000,0.000000
4,Total,100,169.896951,14.674935,4.660838,72.927619,4.342437


### 5.3 Financing stack chart

Stacked bar showing the financing breakdown per track. Compare the CEF grant total with the proposed CEF-E budget (~17 bn EUR for electricity).

In [11]:
stack_df = build_financing_stack_dataset(summary)

fig_stack = px.bar(
    stack_df,
    x="financing_track",
    y="value_beur",
    color="financing_component",
    barmode="stack",
    title="Aggregate Financing Stack by Track (bn EUR)",
    labels={"value_beur": "Financing (bn EUR)", "financing_track": ""},
    category_orders={"financing_track": [TRACK_1, TRACK_2, TRACK_3]},
)
fig_stack.update_layout(height=500)
fig_stack.show()

## Step 6: Sensitivity analysis

The thresholds and assumptions above are inherently uncertain. Here we combine financing assumptions with full-year hourly price windows from 2020 to 2025:

- **Price years:** 2020, 2021, 2022, 2023, 2024, 2025 — changes congestion rents using observed full-year hourly spreads from the local dataset
- **Discount rates:** 4%, 5% (base), 6% — changes annualized CAPEX and therefore all ratios
- **Utilization factors:** 50%, 60% (base), 70% — changes congestion rent estimates
- **Credit thresholds:** 10%, 15% (base), 20% — shifts projects between Track 2 and 3

Only project-year rows with a full overlapping hourly year are kept in this sensitivity table. That yields up to `6 × 3 × 3 × 3 = 162` combinations per project where full-year price data exist.


In [12]:
price_years = range(2020, 2026)
sensitivity_master = build_project_master_table(development_mode=True, price_years=price_years)
sensitivity_master = sensitivity_master[sensitivity_master["hourly_observation_count"].notna()].copy()
sensitivity = build_sensitivity_cases(sensitivity_master, price_scenarios=tuple(f"proxy_{year}" for year in price_years))

print(f"Sensitivity source rows with full-year hourly data: {len(sensitivity_master)}")
print(f"Sensitivity cases: {sensitivity['sensitivity_case'].nunique()}")
print(f"Sensitivity output rows: {len(sensitivity)}")
print()

sensitivity[[
    "project_id", "project_name", "price_scenario", "data_year",
    "hourly_observation_count", "commercial_ratio",
    "credit_constraint_score", "credit_constrained",
]].head(15)


Sensitivity source rows with full-year hourly data: 456
Sensitivity cases: 162
Sensitivity output rows: 12312



,project_id,project_name,price_scenario,data_year,hourly_observation_count,commercial_ratio,credit_constraint_score,credit_constrained
0,4.0,Interconnection Portugal-Spain,proxy_2020,2020,8784.0,0.141188,0.015883,False
1,16.0,Biscay Gulf,proxy_2020,2020,8784.0,0.36097,0.166667,True
2,40.0,Belgium-Luxembourg-Germany: long-term perspective,proxy_2020,2020,8784.0,0.874408,NaN,False
3,47.0,Westtirol (AT) - Vöhringen (DE),proxy_2020,2020,8784.0,16.650408,0.001717,True
4,81.0,North South Interconnector,proxy_2020,2020,8784.0,2.445113,0.051857,True
5,94.0,GerPol Improvements,proxy_2020,2020,8784.0,19.616359,0.011080,True
6,107.0,Celtic Interconnector,proxy_2020,2020,8784.0,0.494421,0.231857,True
7,111.0,Aurora line (3rd AC Finland-Sweden north),proxy_2020,2020,8784.0,2.24393,0.038571,True
8,121.0,Nautilus: multi-purpose interconnector Belgium...,proxy_2020,2020,8784.0,1.023187,0.027027,False
9,124.0,NordBalt phase 2,proxy_2020,2020,8784.0,0.0,NaN,False


## Step 7: Export results

Write all outputs to disk for the policy brief:

- `data/processed/project_master_table.csv` — full project-level table with all metrics and flags
- `data/processed/aggregate_summary.csv` — per-track totals
- `outputs/tables/` — Excel versions for reviewers
- `outputs/charts/` — interactive HTML scatter and stacked-bar charts

All exported tables include assumptions, source provenance, and data-quality flags so reviewers can trace any metric back to its source.

In [13]:
exports = export_outputs(classified)

print("Exported files:")
for name, path in exports.items():
    print(f"  {name}: {path}")

Exported files:
  project_master_table_csv: /Users/lucas/Workspace/PolicyBrief/GridFinancing/data/processed/project_master_table.csv
  aggregate_summary_csv: /Users/lucas/Workspace/PolicyBrief/GridFinancing/data/processed/aggregate_summary.csv
  project_level_results_xlsx: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/tables/project_level_results.xlsx
  aggregate_summary_xlsx: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/tables/aggregate_summary.xlsx
  triage_scatter_html: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/charts/triage_scatter.html
  aggregate_financing_stack_html: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/charts/aggregate_financing_stack.html
